In [2]:
from peft import get_peft_model, LoraConfig, TaskType

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TrainingArguments, Trainer

In [4]:
import pandas as pd
from datasets import Dataset

In [5]:
model_id= 'microsoft/Phi-4-mini-instruct'

In [6]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map='auto')
    return model, tokenizer

In [7]:
def add_lora(model, lora_rank=64):
    config = LoraConfig(
        r=lora_rank,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type=TaskType.CAUSAL_LM,
        lora_dropout=0.1,
        bias="none"
    )
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    return model

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!rm -r "./drive/MyDrive/finetuned"
!mkdir "./drive/MyDrive/finetuned"

In [10]:
def load_table_txt(file_path, sep=None, col_names=None):
    """
    Load a text file structured as a table (not CSV) into a pandas DataFrame.
    Args:
        file_path (str): Path to the text file.
        sep (str or None): Column separator (e.g., '\t', whitespace). If None, uses any whitespace.
        col_names (list or None): Optional list of column names.
    Returns:
        pd.DataFrame: Loaded DataFrame.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Parse and clean lines
    data = []
    for line in lines:
        # Split on tab, strip spaces
        parts = line.strip().split('\t')
        # Remove empty strings and strip whitespace
        parts = [p.strip() for p in parts if p.strip()]
        # Only take rows with 2 elements
        if len(parts) == 2:
            data.append(parts)

    # Create DataFrame
    df = pd.DataFrame(data, columns=col_names)

    # Display the result
    return df

In [11]:
dataset = load_table_txt(r"./drive/MyDrive/data.txt", sep=None, col_names=["DHS Code", "Description"])

In [11]:
model, tokenizer = load_model(model_id)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [12]:
tokenizer.device = (model.device)

In [13]:
tokenizer.device

device(type='cuda', index=0)

In [17]:
def prep_dataset(model, tokenizer, df, out_dir, save_every=50):
    """Simple version with counter and periodic saving"""

    generation_args = {
        "max_new_tokens": 200,
        "temperature": 0.3,
        "do_sample": True,
        "pad_token_id": tokenizer.eos_token_id,
    }

    # Check if progress file exists and resume from there
    try:
        existing_df = pd.read_csv(out_dir + 'progress.csv')
        start_from = len(existing_df)
        explanations = existing_df['explanation'].tolist()
        print(f"Found existing progress.csv with {start_from} completed rows")

        if start_from >= len(df):
            print("All rows already completed! Check progress.csv")
            return existing_df

        print(f"Resuming from row {start_from + 1}")
    except FileNotFoundError:
        start_from = 0
        explanations = []
        print(f"Starting fresh - processing {len(df)} rows, saving every {save_every} iterations...")

    counter = start_from

    for index, row in df.iloc[start_from:].iterrows():
        try:
            counter += 1
            code, desc = row["DHS Code"], row["Description"]

            # Create prompt
            messages = [
                {"role": "system", "content": "You are a medical expert. Explain medical conditions simply in ** Less than 250 characters **."},
                {"role": "user", "content": f"Explain the following: {desc}"}
            ]

            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)

            # Generate
            with torch.no_grad():
                outputs = model.generate(inputs.input_ids, **generation_args)

            # Extract explanation
            new_tokens = outputs[0][inputs.input_ids.shape[1]:]
            explanation = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            explanations.append(explanation)

            print(f"Iteration {counter}: {code} -> {explanation[:50]}...")

            # Save every N iterations (same file)
            if counter % save_every == 0:
                temp_df = df.iloc[:counter].copy()
                temp_df['explanation'] = explanations
                temp_df.to_csv(out_dir + 'progress.csv', index=False)
                print(f"Saved progress at iteration {counter}")

        except Exception as e:
            print(f"Error at iteration {counter}: {e}")
            explanations.append("Error generating explanation")

    # Final save
    df_copy = df.copy()
    df_copy['explanation'] = explanations
    df_copy.to_csv('final_results.csv', index=False)
    print(f"Completed! Final results saved to final_results.csv")

    return df_copy

# Usage:
# First run: results = prep_dataset_simple(model, tokenizer, df, save_every=25)
# If interrupted, just run again: results = prep_dataset_simple(model, tokenizer, df, save_every=25)
# It will automatically resume from where it left off!

In [16]:
# def format_data(df):
#     return {
#         "input_ids": tokenizer(df['DHS Code'], return_tensors= 'pt', truncation=True, max_length=512),
#         "labels": tokenizer(df['Description'], return_tensors= 'pt', truncation=True, max_length=512)
#     }

In [21]:
data = prep_dataset(model, tokenizer, dataset, f"./drive/MyDrive/", 50)

Found existing progress.csv with 10 completed rows
Resuming from row 11
Iteration 11: 86960 -> Reduced blood volume can lead to dehydration, affe...
Iteration 12: 86965 -> Pooling blood platelets is a process where platele...
Iteration 13: 86985 -> Splitting blood involves separating whole blood in...
Iteration 14: 0011M -> OncPrst8, a protein, is encoded by the Onc12 gene....
Iteration 15: 0012M -> The Onc mRNA 5 gene encodes a kinase that, when mu...
Iteration 16: 0013M -> Oncogene mutations in the 5th chromosome can lead ...
Iteration 17: 0016M -> This refers to a genetic mutation (MRNA 219) in th...
Iteration 18: 0017M -> Anaplastic large cell lymphoma (ALCL) is a type of...
Iteration 19: 0019M -> Cystic fibrosis is a genetic disorder affecting th...
Iteration 20: 0020M -> Oncogenes are genes that can cause cancer when mut...
Iteration 21: 0640T -> Non-compliance with Infection Control Standards Ou...
Iteration 22: 0859T -> Non-coding RNA (ncRNA) interferes with the splicin...
Iter

In [14]:
model.device

device(type='cuda', index=0)

In [15]:
model = add_lora(model)

trainable params: 35,651,584 || all params: 3,871,673,344 || trainable%: 0.9208


In [16]:
model.device

device(type='cuda', index=0)

In [17]:
new_dataset = pd.read_csv(r"./drive/MyDrive/progress.csv")

In [18]:
dataset = Dataset.from_pandas(new_dataset)

In [19]:
dataset

Dataset({
    features: ['DHS Code', 'Description', 'explanation'],
    num_rows: 1300
})

In [20]:
print(new_dataset['Description'].str.len().median())

26.0


In [21]:
def format_batch(batch):
    prompts = [f"<|user|>Given the medical code: {code}, and its {description} provide an explanation in simple english.<|assistant|>" for code, description in zip(batch['DHS Code'], batch['Description'])]
    responses = [f"The code represents {explanation}." for explanation in batch['explanation']]

    # Tokenize inputs and labels as a batch
    input_enc = tokenizer(
        prompts,
        padding="max_length",
        truncation=True,
        max_length=153
    )

    label_enc = tokenizer(
        responses,
        padding="max_length",
        truncation=True,
        max_length=153
    )

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": label_enc["input_ids"]
    }

tokenized_dataset = dataset.map(format_batch, batched=True, remove_columns=dataset.column_names)


Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

In [22]:
len(tokenized_dataset)

1300

In [23]:
tokenized_dataset[0]

{'input_ids': [200021,
  39520,
  290,
  7774,
  3490,
  25,
  220,
  46991,
  6283,
  11,
  326,
  1617,
  23050,
  120596,
  827,
  315,
  3587,
  448,
  30547,
  306,
  4705,
  37785,
  13,
  200019,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  199999,
  19999

In [24]:
dataset = tokenized_dataset

In [25]:
def train(
            model,
            output_dir,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=50,
            max_steps=500,
            learning_rate=1e-4,
            bf16=True,
            logging_steps=5,
            save_steps=100,
            save_total_limit=2,
            optim='adamw_torch_fused',
            evaluation_strategy="no"
          ):
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=learning_rate,
        bf16=bf16,
        logging_steps=logging_steps,
        save_steps=save_steps,
        save_total_limit=save_total_limit,
        optim=optim,
        # evaluation_strategy=evaluation_strategy,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer
    )
    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)

In [26]:
train(model=model, output_dir=r'./drive/MyDrive/finetuned/')

/tmp/ipython-input-25-2907713612.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chiragml (chiragml-civicai-lab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
5,11.232400
10,10.977000
15,9.686400
20,5.723000
25,3.583900
30,2.831100
35,2.730100
40,2.272300
45,1.976300
50,1.838600


Step,Training Loss
5,11.232400
10,10.977000
15,9.686400
20,5.723000
25,3.583900
30,2.831100
35,2.730100
40,2.272300
45,1.976300
50,1.838600


In [27]:
from peft import PeftModel, PeftConfig

In [28]:
def load_finetuned_model(adapter_path, base_model):

  # Load the adapter config to get base model info
  peft_config = PeftConfig.from_pretrained(adapter_path)

  # Load base model
  base_model = AutoModelForCausalLM.from_pretrained(
      peft_config.base_model_name_or_path,
      torch_dtype=torch.bfloat16,
      device_map='auto',
      trust_remote_code=True
  )

  # Load and apply adapters
  model = PeftModel.from_pretrained(base_model, adapter_path)

  # Load tokenizer
  tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)
  if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token

  return model, tokenizer

In [31]:
def generate_response(model, tokenizer, medical_code, max_length=256):
    """Generate response for a given medical code"""
    code = str(medical_code)
    description = new_dataset[new_dataset['DHS Code'] == code]['Description'].values[0]
    # Format the prompt like during training
    prompt = f"<|user|>Given the medical code: {code}, and its {description} provide an explanation in simple english.<|assistant|>"

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response)
    # Extract just the assistant's response
    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[1].strip()

    return response

In [32]:
ft_model, ft_tokenizer = load_finetuned_model(r"./")

TypeError: load_finetuned_model() missing 1 required positional argument: 'base_model'

In [ ]:
generate_response(model, tokenizer, 76510)

In [ ]:
# =============================================================================
# DEBUGGING STEPS FOR MODEL GENERATION ISSUES
# =============================================================================

def debug_model_generation(model, tokenizer, medical_code="76510"):
    """Comprehensive debugging for generation issues"""

    print("🔍 DEBUGGING MODEL GENERATION")
    print("=" * 60)

    # Step 1: Check model state
    print("1. MODEL STATE CHECK:")
    print("-" * 30)
    if hasattr(model, 'peft_config'):
        print("✅ Model has LoRA adapters")
        model.print_trainable_parameters()
    else:
        print("❌ Model has NO LoRA adapters - this might be the issue!")

    # Step 2: Test different prompt formats
    print("\n2. TESTING DIFFERENT PROMPT FORMATS:")
    print("-" * 30)

    prompts_to_test = [
        # Format 1: Your current format
        f"Given the medical code: {medical_code}, provide its description.",

        # Format 2: With proper Phi-4 chat format
        f"<|user|>\nGiven the medical code: {medical_code}, provide its description.\n<|assistant|>\n",

        # Format 3: Simple format
        f"Medical code {medical_code} means:",

        # Format 4: Instruction format
        f"<|system|>\nYou are a medical coding expert.\n<|user|>\nWhat does medical code {medical_code} mean?\n<|assistant|>\n",
    ]

    for i, prompt in enumerate(prompts_to_test, 1):
        print(f"\nFormat {i}: {prompt[:50]}...")
        response = generate_with_debug(model, tokenizer, prompt)
        print(f"Response: {response[:100]}...")
        print("-" * 20)

    # Step 3: Check tokenization
    print("\n3. TOKENIZATION CHECK:")
    print("-" * 30)
    test_prompt = f"<|user|>\nGiven the medical code: {medical_code}, provide its description.\n<|assistant|>\n"

    inputs = tokenizer(test_prompt, return_tensors="pt")
    print(f"Input length: {inputs['input_ids'].shape[1]} tokens")
    print(f"Input tokens: {inputs['input_ids'][0][:10].tolist()}...")

    # Decode to check tokenization
    decoded = tokenizer.decode(inputs['input_ids'][0])
    print(f"Decoded input: {decoded}")

    # Step 4: Test with base model for comparison
    print("\n4. COMPARISON WITH BASE MODEL:")
    print("-" * 30)
    try:
        # Load base model for comparison
        base_model = AutoModelForCausalLM.from_pretrained(
            "microsoft/Phi-4-mini-instruct",
            torch_dtype=torch.bfloat16,
            device_map='auto',
            load_in_4bit=True
        )

        print("Testing base model response:")
        base_response = generate_with_debug(base_model, tokenizer, test_prompt)
        print(f"Base model: {base_response[:100]}...")

        print("Testing your fine-tuned model:")
        ft_response = generate_with_debug(model, tokenizer, test_prompt)
        print(f"Fine-tuned: {ft_response[:100]}...")

    except Exception as e:
        print(f"Couldn't load base model for comparison: {e}")

def generate_with_debug(model, tokenizer, prompt, max_length=256):
    """Generate text with detailed debugging"""

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]

    # Generate with multiple parameter sets
    generation_configs = [
        {"temperature": 0.7, "do_sample": True, "top_p": 0.9},
        {"temperature": 0.3, "do_sample": True, "top_p": 0.8},
        {"do_sample": False},  # Greedy decoding
        {"temperature": 1.0, "do_sample": True, "top_k": 50},
    ]

    for i, config in enumerate(generation_configs):
        try:
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_length=min(max_length, input_length + 100),
                    num_return_sequences=1,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    **config
                )

            # Decode response
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Remove input from response
            if prompt in response:
                response = response.replace(prompt, "").strip()

            print(f"Config {i+1} {config}: {response[:50]}...")

            if response and response != prompt:
                return response

        except Exception as e:
            print(f"Config {i+1} failed: {e}")

    return "No valid response generated"

In [ ]:
debug_model_generation(model, tokenizer)

🔍 DEBUGGING MODEL GENERATION
1. MODEL STATE CHECK:
------------------------------
✅ Model has LoRA adapters
trainable params: 8,912,896 || all params: 3,844,934,656 || trainable%: 0.2318

2. TESTING DIFFERENT PROMPT FORMATS:
------------------------------

Format 1: Given the medical code: 76510, provide its descrip...
Config 1 {'temperature': 0.7, 'do_sample': True, 'top_p': 0.9}: ...
Config 2 {'temperature': 0.3, 'do_sample': True, 'top_p': 0.8}: ...
Config 3 {'do_sample': False}: ...
Config 4 {'temperature': 1.0, 'do_sample': True, 'top_k': 50}: ...
Response: No valid response generated...
--------------------

Format 2: <|user|>
Given the medical code: 76510, provide it...
Config 1 {'temperature': 0.7, 'do_sample': True, 'top_p': 0.9}: Given the medical code: 76510, provide its descrip...
Response: Given the medical code: 76510, provide its description.
...
--------------------

Format 3: Medical code 76510 means:...
Config 1 {'temperature': 0.7, 'do_sample': True, 'top_p': 0.9}: .

In [ ]:
def fix_generation_issues(model, tokenizer):
    """Apply common fixes for generation issues"""

    print("🔧 APPLYING COMMON FIXES:")
    print("=" * 40)

    # Fix 1: Ensure proper tokenizer setup
    print("1. Fixing tokenizer setup...")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
        print("   ✅ Set pad_token = eos_token")

    if tokenizer.chat_template is None:
        print("   ⚠️  No chat template found")

    # Fix 2: Check if model is in eval mode
    print("2. Setting model to eval mode...")
    model.eval()
    print("   ✅ Model set to eval mode")

    # Fix 3: Test with different generation approach
    print("3. Testing improved generation function...")

    def improved_generate(medical_code):
        # Use the exact format from training
        prompt = f"<|user|>\nGiven the medical code: {medical_code}, provide its description.\n<|assistant|>\n"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=50,  # Generate only new tokens
                min_new_tokens=5,   # Ensure some generation
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Get only the new tokens (remove input)
        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        return response.strip()

    # Test the improved function
    test_response = improved_generate("76510")
    print(f"   Test response: {test_response}")

    return improved_generate

In [ ]:
fix_generation_issues(model, tokenizer)

🔧 APPLYING COMMON FIXES:
1. Fixing tokenizer setup...
2. Setting model to eval mode...
   ✅ Model set to eval mode
3. Testing improved generation function...
   Test response: .x).st-


<function __main__.fix_generation_issues.<locals>.improved_generate(medical_code)>

In [ ]:
def improved_generate(medical_code):
        # Use the exact format from training
        prompt = f"<|user|>\nGiven the medical code: {medical_code}, provide its description.\n<|assistant|>\n"

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=50,  # Generate only new tokens
                min_new_tokens=5,   # Ensure some generation
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Get only the new tokens (remove input)
        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        return response.strip()

In [ ]:
improved_generate(76501)

'-1ct c&'

In [ ]:
def check_training_data_format(dataset_path):
    """Check if training data format matches inference format"""

    print("📊 CHECKING TRAINING DATA FORMAT:")
    print("=" * 40)

    try:
        # Load your training data
        with open(dataset_path, 'r') as f:
            lines = f.readlines()[:5]  # Check first 5 lines

        print("First 5 lines of training data:")
        for i, line in enumerate(lines, 1):
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                code, description = parts[0], parts[1]
                print(f"{i}. Code: {code} | Description: {description[:50]}...")

        # Show what the training prompt should look like
        print("\nTraining prompt should be:")
        example_code = lines[0].strip().split('\t')[0] if lines else "76510"
        example_desc = lines[0].strip().split('\t')[1] if lines and len(lines[0].strip().split('\t')) > 1 else "Example description"

        training_prompt = f"<|user|>\nGiven the medical code: {example_code}, provide its description.\n<|assistant|>\n{example_desc}<|end|>"
        print(training_prompt)

    except Exception as e:
        print(f"Couldn't check training data: {e}")

In [ ]:
check_training_data_format(r'./data.txt')

📊 CHECKING TRAINING DATA FORMAT:
First 5 lines of training data:
1. Code: 86152 | Description: Cell enumeration &id...
2. Code: 86153 | Description: Cell enumeration phys interp...
3. Code: 86890 | Description: Autologous blood process...
4. Code: 86891 | Description: Autologous blood op salvage...
5. Code: 86927 | Description: Plasma fresh frozen...

Training prompt should be:
<|user|>
Given the medical code: 86152, provide its description.
<|assistant|>
Cell enumeration &id<|end|>
